In [ ]:
import mne
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from glob import glob
import scipy.io
import h5py
import os
from tqdm import tqdm
import re

## Concatenate TF

In [ ]:
session = "s2_r1"

# csv_path = f"derivatives/behavior/{session}/"
output_path = f"/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/preprocessed/TF_arrays/{session}/"

# take IDs from fully processed behavioral data (checked for accuracy, validRT, missed responses) separately for each condition
# sub_nonsoc = list(pd.read_csv(glob(f"{csv_path}thrive_data_nonsoc.csv")[0])["sub"])
# sub_soc = list(pd.read_csv(glob(f"{csv_path}thrive_data_soc.csv")[0])["sub"])

# Define the regex pattern to match 'sub-' followed by digits
pattern = r'sub-\d+'

# tf_files = sorted(glob(f"{data_path}/sub-*{condition}*.mat"))
for measure in [
    "TF",
    "ITPS",
    "ICPS",
    "wPLI"
]:
    print(f"Working on {measure} ... ")
    if measure == "ITPS" or measure == "ICPS":
        key_idx = 1
    else:
        key_idx = -1
    data_path = f"/home/data/NDClab/analyses/thrive-theta-ddm/derivatives/preprocessed/TF_outputs/{session}/resp/{measure}/"
    for condition in tqdm(["resp_ns_c_1", "resp_ns_i_0", "resp_ns_i_1",
                           "resp_s_i_1", "resp_s_c_1", "resp_s_i_0"]):
        
        arr_list = []
        subjects_with_data = []
        # Extract subject ids that have TF data
        matched_parts = [
            re.search(pattern, s).group(0) if re.search(pattern, s) else None for s in glob(
                f"{data_path}/sub-*all_eeg_processed_data*{measure}*{condition}*.mat"
            )
        ]
        for sub_id in sorted(matched_parts):
            # check if there is data for that subject for that condition
            try:
                # sort all TF files by sub_id first
                tf_files = sorted(glob(f"{data_path}/{sub_id}*all_eeg_processed_data*{measure}*{condition}*.mat"))
                assert len(tf_files) == 1, "Check your tf_files length!"

                # read TF array
                data_file = h5py.File(tf_files[0])
                key_list = list(data_file.keys())
                data = data_file[key_list[key_idx]]
                # take only actual data with channels * times * freqs
                assert data.shape == (64, 375, 59), "Check your data!"
                arr_list.append(data)
                subjects_with_data.append(sub_id) # that way, the actual data and participant ids will go in the same order
            except: continue
        
        # concatenate all valid subject data for a given condition 
        full_data = np.stack(arr_list, axis=0)
        # make sure the number of arrays is the same as number of subject ids saved previously
        assert full_data.shape[0] == len(subjects_with_data), "Check your data!"
        print(f"# of subjects to have condition {condition}: {len(arr_list)}")
        # save resulting subs * channels * times * freqs
        scipy.io.savemat(f"{output_path}/{measure}_{condition}.mat",
                         {
                             f"{measure}_{condition}": full_data,
                             f"subjects": subjects_with_data,
                         })

In [ ]:
tf_files

In [ ]:
glob("/derivatives/preprocessed/TF_outputs/main/resp/*")

In [ ]:
!ls

In [ ]:
f"{data_path}/{sub_id}*all_eeg_processed_data*{measure}*{condition}*.mat"

## Inspect number of events

In [ ]:
import numpy as np

def eeg_point2lat(lat_array, epoch_array, srate, timewin=None, timeunit=1):
    """
    Convert latency in data points to latency in ms relative to the time locking.

    Parameters:
    lat_array (array-like): Latency array in data points assuming concatenated data epochs.
    epoch_array (array-like or None): Epoch number corresponding to each latency value.
    srate (float): Data sampling rate in Hz.
    timewin (list or None): [min, max] time limits in 'timeunit' units. Default is None.
    timeunit (float): Time unit in seconds. Default is 1 (seconds).

    Returns:
    numpy.ndarray: Converted latency values (in 'timeunit' units) for each epoch.
    """
    
    lat_array = np.array(lat_array)
    
    if epoch_array is None:
        epoch_array = np.ones_like(lat_array)
    else:
        epoch_array = np.array(epoch_array)

    if timewin is None:
        timewin = [0, 0]
    
    if len(lat_array) != len(epoch_array):
        if len(epoch_array) != 1:
            raise ValueError("Latency and epoch arrays must have the same length")
        else:
            epoch_array = np.ones_like(lat_array) * epoch_array[0]
    
    if len(timewin) != 2:
        raise ValueError("Timelimits array must have length 2")
    
    timewin = np.array(timewin) * timeunit
    
    if len(timewin) == 2:
        pnts = int((timewin[1] - timewin[0]) * srate + 1)
        pnts = (timewin[1] - timewin[0]) * srate + 1
    else:
        pnts = 0
    
    newlat = ((lat_array - (epoch_array - 1) * pnts - 1) / srate + timewin[0]) / timeunit
    
    return np.round(newlat * 1e9) / 1e9

In [ ]:
import time
sub_to_inspect = "114"
trial_data = dict({
        "sub": [],
        "s_resp_incon_error": [],
        "s_resp_incon_corr": [],
        "ns_resp_incon_error": [],
        "ns_resp_incon_corr": [],
        "s_stim_incon_corr": [],
        "s_stim_con_corr": [],
        "ns_stim_incon_corr": [],
        "ns_stim_con_corr": [],
})

dataset_path = "/home/data/NDClab/datasets/thrive-dataset/"

sub_ids = sorted([i.split("/")[-1] for i in glob(
        f"{dataset_path}derivatives/preprocessed/sub-*{sub_to_inspect}*")])

list_of_eeg_file = sorted(
    glob(
        f"{dataset_path}derivatives/preprocessed/*{sub_to_inspect}*/s1_r1/eeg/*all*eeg*processed*data*.set")
)

start = time.time()

for file_idx, filename in enumerate(list_of_eeg_file):
    sub_id = sub_ids[file_idx].split("-")[-1]
    trial_data["sub"].append(sub_id)
    EEG = scipy.io.loadmat(filename, squeeze_me=True, struct_as_record=False)["EEG"]
    EEG_mne = mne.io.read_epochs_eeglab(filename, verbose = 'ERROR',)
    
    events = EEG.event
    n_times = EEG.pnts
    sr = EEG.srate
    num_ch = EEG.nbchan

    drop_idx = []
    for i in range(len(events)):
        latency = eeg_point2lat(
            [events[i].latency],
            [events[i].epoch],
            sr,
            timewin = [EEG.xmin*1000, EEG.xmax*1000],
            timeunit = 1e-3,
             )
        if latency >= -.1 and latency <= .1:
            drop_idx.append(i)
    
    events = [ev for ev in events if list(events).index(ev) in drop_idx]
    print(f"sub-{sub_id}: {len(events)} good events were found!")
    
    trial_data["s_resp_incon_error"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 0) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["s_resp_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_resp_incon_error"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 0) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_resp_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["s_stim_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "stim") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["s_stim_con_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "stim") & (ev.congruency == "c")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_stim_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "stim") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_stim_con_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "stim") & (ev.congruency == "c")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))

end = time.time()
print(f"Executed time {np.round(end - start, 2)} s")

pd.DataFrame(trial_data)